# Skill Execution Quality Benchmark

**The most important benchmark:** does AIF's semantic structure help LLMs actually follow instructions better?

This benchmark tests whether typed blocks (`@step`, `@verify`, `@precondition`) improve step coverage, constraint respect, and output contract adherence. This is the core value proposition — **instruction following matters more than token savings.**

For each skill + scenario, the LLM is given the skill in multiple formats (Raw Markdown, LML Aggressive, HTML, JSON IR) and asked to execute it. A separate judge LLM then scores how well the response follows the skill's steps, respects constraints, and meets the output contract.

Requires `ANTHROPIC_API_KEY` environment variable.

## Setup

In [ ]:
import json
import os
import subprocess
import sys
import time
import math
from pathlib import Path

import anthropic

MODEL_EXECUTOR = "claude-sonnet-4-6"
MODEL_JUDGE = "claude-sonnet-4-6"
PROJECT_ROOT = Path(os.path.abspath("")).parent.parent
AIF_CLI = PROJECT_ROOT / "target" / "release" / "aif-cli"

FORMATS = [
    ("raw_md", "Raw Markdown", "export"),
    ("lml_aggressive", "LML Aggressive", "lml-aggressive"),
    ("html", "HTML", "html"),
    ("json_ir", "JSON IR", "json"),
]

# Init Anthropic client
api_key = os.environ.get("ANTHROPIC_API_KEY")
if not api_key:
    raise RuntimeError("ANTHROPIC_API_KEY not set")

client = anthropic.Anthropic(api_key=api_key)

print(f"Executor:     {MODEL_EXECUTOR}")
print(f"Judge:        {MODEL_JUDGE}")
print(f"Formats:      {len(FORMATS)}")
print(f"Project root: {PROJECT_ROOT}")
print(f"AIF CLI:      {AIF_CLI}")
print(f"CLI exists:   {AIF_CLI.exists()}")

## Scenarios

Five test scenarios across two skill categories, spanning easy/medium/hard difficulty levels.

| # | Scenario | Category | Difficulty | What it tests |
|---|----------|----------|------------|---------------|
| 1 | SQL Injection Bug | code-review | easy | Obvious vulnerability -- all formats should catch it |
| 2 | Clean Code (Should Approve) | code-review | hard | Good code -- tests whether model over-flags |
| 3 | Race Condition in Counter | code-review | medium | Concurrency bug requiring understanding of shared state |
| 4 | eval() in User Input | security | easy | Classic eval injection -- all formats should catch it |
| 5 | Shell Injection via Template Literal | security | medium | Shell injection hidden in a template literal |

In [ ]:
SCENARIOS = [
    # -- Code Review: Obvious Bug --
    {
        "skill_file": "examples/skills/code_review.aif",
        "name": "Code Review: SQL Injection Bug",
        "category": "code-review",
        "difficulty": "easy",
        "description": "Obvious security vulnerability -- all formats should catch it",
        "prompt": (
            "Please review this code change:\n\n"
            "```python\n"
            "# auth.py - login endpoint\n"
            "def login(request):\n"
            "    email = request.POST['email']\n"
            "    password = request.POST['password']\n"
            "    query = f\"SELECT * FROM users WHERE email='{email}' AND password='{password}'\"\n"
            "    user = db.execute(query).fetchone()\n"
            "    if user:\n"
            "        return create_session(user)\n"
            "    return HttpResponse('Invalid credentials', status=401)\n"
            "```\n"
        ),
        "expected_steps": [
            "understand context or intent of the change",
            "identify the SQL injection vulnerability",
            "suggest parameterized query as fix",
            "categorize as blocking issue",
        ],
        "expected_constraints": [
            "should NOT approve without flagging the injection",
            "should provide a concrete fix, not just say 'fix it'",
            "should distinguish blocking from suggestion",
        ],
        "output_contract": "structured review with blocking/suggestion/praise categories",
    },

    # -- Code Review: Subtle Judgment Call --
    {
        "skill_file": "examples/skills/code_review.aif",
        "name": "Code Review: Clean Code (Should Approve)",
        "category": "code-review",
        "difficulty": "hard",
        "description": "Good code that should be approved -- tests whether model over-flags",
        "prompt": (
            "Please review this code change:\n\n"
            "```rust\n"
            "/// Parse a comma-separated list of IDs, skipping invalid entries.\n"
            "pub fn parse_ids(input: &str) -> Vec<u64> {\n"
            "    input\n"
            "        .split(',')\n"
            "        .filter_map(|s| s.trim().parse::<u64>().ok())\n"
            "        .collect()\n"
            "}\n\n"
            "#[cfg(test)]\n"
            "mod tests {\n"
            "    use super::*;\n\n"
            "    #[test]\n"
            "    fn parses_valid_ids() {\n"
            "        assert_eq!(parse_ids(\"1, 2, 3\"), vec![1, 2, 3]);\n"
            "    }\n\n"
            "    #[test]\n"
            "    fn skips_invalid() {\n"
            "        assert_eq!(parse_ids(\"1, abc, 3\"), vec![1, 3]);\n"
            "    }\n"
            "}\n"
            "```\n"
        ),
        "expected_steps": [
            "understand the code's purpose",
            "check correctness -- no bugs to find",
            "note good patterns (tests, documentation, idiomatic Rust)",
        ],
        "expected_constraints": [
            "should NOT flag non-issues or bikeshed on style",
            "should approve or approve with minor comments only",
            "should mention the good test coverage as praise",
        ],
        "output_contract": "structured review that approves the clean code without false blocking issues",
    },

    # -- Code Review: Race Condition --
    {
        "skill_file": "examples/skills/code_review.aif",
        "name": "Code Review: Race Condition in Counter",
        "category": "code-review",
        "difficulty": "medium",
        "description": "Concurrency bug -- requires understanding shared mutable state",
        "prompt": (
            "Please review this code change:\n\n"
            "```python\n"
            "# counter.py - request counter for rate limiting\n"
            "import threading\n\n"
            "request_count = 0\n\n"
            "def increment():\n"
            "    global request_count\n"
            "    current = request_count\n"
            "    request_count = current + 1\n\n"
            "def get_count():\n"
            "    return request_count\n\n"
            "def is_rate_limited():\n"
            "    return request_count > 1000\n"
            "```\n"
        ),
        "expected_steps": [
            "understand the code's purpose (rate limiting counter)",
            "identify the race condition (read-modify-write without lock)",
            "suggest fix using threading.Lock or atomic counter",
            "categorize as blocking",
        ],
        "expected_constraints": [
            "should identify the TOCTOU race condition",
            "should provide a concrete fix with Lock or atomic",
            "should NOT just say 'add locking' without showing how",
        ],
        "output_contract": "structured review identifying the race condition as a blocking issue with concrete fix",
    },

    # -- Security: eval() --
    {
        "skill_file": "examples/skills/security-guidance.aif",
        "name": "Security: Detect eval() in User Input",
        "category": "security",
        "difficulty": "easy",
        "description": "Classic eval injection -- all formats should catch it",
        "prompt": (
            "I'm writing a calculator feature. Here's my code:\n\n"
            "```javascript\n"
            "app.post('/calculate', (req, res) => {\n"
            "    const expression = req.body.expression;\n"
            "    const result = eval(expression);\n"
            "    res.json({ result });\n"
            "});\n"
            "```\n"
        ),
        "expected_steps": [
            "identify eval() with user input as a security risk",
            "explain the vulnerability (arbitrary code execution)",
            "suggest a safe alternative (math parser library like mathjs)",
        ],
        "expected_constraints": [
            "must flag eval() as critical/high severity",
            "must provide a concrete safe replacement, not just 'don't use eval'",
        ],
        "output_contract": "security finding with category, severity, and safe replacement code",
    },

    # -- Security: Subtle Shell Injection --
    {
        "skill_file": "examples/skills/security-guidance.aif",
        "name": "Security: Shell Injection via Template Literal",
        "category": "security",
        "difficulty": "medium",
        "description": "Shell injection hidden in a template literal -- tests depth of analysis",
        "prompt": (
            "Here's a file search utility I'm building:\n\n"
            "```python\n"
            "import subprocess\n\n"
            "def search_logs(query, log_dir='/var/log'):\n"
            "    cmd = f'grep -r \"{query}\" {log_dir}'\n"
            "    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)\n"
            "    return result.stdout.splitlines()\n"
            "```\n"
        ),
        "expected_steps": [
            "identify shell=True with user-controlled input as shell injection",
            "explain the attack vector (query could contain shell metacharacters)",
            "suggest subprocess.run with list args instead of shell=True",
        ],
        "expected_constraints": [
            "must identify shell=True as the vulnerability, not just f-string",
            "must provide subprocess.run(['grep', '-r', query, log_dir]) as fix",
        ],
        "output_contract": "security finding identifying shell injection with concrete safe subprocess.run replacement",
    },
]

print(f"Defined {len(SCENARIOS)} scenarios:")
for i, s in enumerate(SCENARIOS, 1):
    print(f"  {i}. [{s['difficulty']:6s}] {s['name']}")

## Metrics

Four metrics measure how well an LLM follows a skill:

| Metric | What it measures | Why it matters |
|--------|-----------------|----------------|
| **Step coverage** | Fraction of `@step` blocks reflected in the response | Did the LLM actually do each step the skill prescribes? |
| **Constraint respect** | Fraction of `@red_flag` / `@verify` items honored | Did the LLM avoid the things the skill warns against? |
| **Output contract** | Does the response match `@output_contract` criteria | Is the response structured the way the skill demands? |
| **Overall compliance** | Weighted average of the above | Single score for ranking |

**Why these matter:** an LLM that misses steps or violates constraints is worse than one that follows them all, regardless of token count. The goal is not to find the cheapest format -- it is to find the format that produces the most compliant execution.

## Helper Functions

In [ ]:
def stddev(values: list) -> float:
    """Sample standard deviation."""
    if len(values) < 2:
        return 0.0
    mean = sum(values) / len(values)
    variance = sum((v - mean) ** 2 for v in values) / (len(values) - 1)
    return math.sqrt(variance)


def compile_skill(skill_path: str, fmt: str) -> str:
    """Compile an AIF skill to a target format using aif-cli."""
    if fmt == "export":
        result = subprocess.run(
            [str(AIF_CLI), "skill", "export", skill_path],
            capture_output=True, text=True, timeout=15,
        )
    else:
        result = subprocess.run(
            [str(AIF_CLI), "compile", skill_path, "--format", fmt],
            capture_output=True, text=True, timeout=15,
        )
    if result.returncode != 0:
        return f"[compilation failed: {result.stderr.strip()[:200]}]"
    return result.stdout


def execute_skill(client, skill_text: str, user_prompt: str) -> tuple:
    """Have the executor model follow the skill. Returns (response_text, latency_s)."""
    system = (
        "You are an AI assistant. Follow the skill/instructions below precisely. "
        "Apply them to the user's request.\n\n"
        "=== SKILL ===\n" + skill_text + "\n=== END SKILL ==="
    )
    t0 = time.time()
    response = client.messages.create(
        model=MODEL_EXECUTOR,
        max_tokens=2048,
        system=system,
        messages=[{"role": "user", "content": user_prompt}],
    )
    latency = time.time() - t0
    return response.content[0].text, latency


def judge_compliance(
    client,
    skill_text: str,
    response_text: str,
    expected_steps: list,
    expected_constraints: list,
    output_contract: str,
) -> dict:
    """Have the judge model score the executor's compliance."""
    prompt = f"""You are evaluating whether an AI assistant correctly followed a skill/instruction set.

SKILL (reference -- first 3000 chars):
{skill_text[:3000]}

ASSISTANT'S RESPONSE (first 3000 chars):
{response_text[:3000]}

EXPECTED STEPS (did the response cover these?):
{json.dumps(expected_steps)}

EXPECTED CONSTRAINTS (were these respected?):
{json.dumps(expected_constraints)}

OUTPUT CONTRACT:
{output_contract}

Score each dimension from 0.0 to 1.0. Be precise -- 0.5 means half the items were covered.

Respond with ONLY this JSON (no other text, no markdown fences):
{{"step_coverage": <float>, "step_details": "<which steps covered/missed>", "constraint_respect": <float>, "constraint_details": "<which constraints respected/violated>", "output_contract_met": <float>, "output_contract_details": "<how well output matches contract>", "overall": <float>}}"""

    response = client.messages.create(
        model=MODEL_JUDGE,
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}],
    )
    text = response.content[0].text.strip()

    # Extract JSON from response (handle markdown code blocks)
    if "```" in text:
        text = text.split("```")[1]
        if text.startswith("json"):
            text = text[4:]
        text = text.strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return {
            "step_coverage": 0.0, "constraint_respect": 0.0,
            "output_contract_met": 0.0, "overall": 0.0,
            "parse_error": text[:300],
        }


print("Helper functions defined.")

## Run Benchmark

For each scenario x format: compile skill, count tokens, execute, judge, collect results.

In [ ]:
all_results = []

print("=" * 80)
print("AIF Skill Execution Quality Benchmark")
print(f"Executor: {MODEL_EXECUTOR} | Judge: {MODEL_JUDGE}")
print(f"Scenarios: {len(SCENARIOS)} | Formats: {len(FORMATS)}")
print("=" * 80)

for scenario in SCENARIOS:
    print(f"\nScenario: {scenario['name']}")
    print(f"  Category: {scenario['category']} | Difficulty: {scenario['difficulty']}")
    print(f"  {scenario['description']}")

    for fmt_key, fmt_label, fmt_arg in FORMATS:
        skill_text = compile_skill(
            str(PROJECT_ROOT / scenario["skill_file"]), fmt_arg
        )
        if "[compilation failed" in skill_text:
            print(f"  {fmt_label:20s}  SKIP (compilation failed)")
            continue

        skill_tokens = client.messages.count_tokens(
            model=MODEL_EXECUTOR,
            messages=[{"role": "user", "content": skill_text}],
        ).input_tokens

        # Execute
        print(f"  {fmt_label:20s}  executing...", end="", flush=True)
        response_text, exec_time = execute_skill(client, skill_text, scenario["prompt"])

        # Judge
        print(f" judging...", end="", flush=True)
        t0 = time.time()
        scores = judge_compliance(
            client, skill_text, response_text,
            scenario["expected_steps"],
            scenario["expected_constraints"],
            scenario["output_contract"],
        )
        judge_time = time.time() - t0

        result = {
            "scenario": scenario["name"],
            "category": scenario["category"],
            "difficulty": scenario["difficulty"],
            "format": fmt_label,
            "format_key": fmt_key,
            "skill_tokens": skill_tokens,
            "step_coverage": scores.get("step_coverage", 0),
            "constraint_respect": scores.get("constraint_respect", 0),
            "output_contract_met": scores.get("output_contract_met", 0),
            "overall": scores.get("overall", 0),
            "exec_time_s": round(exec_time, 1),
            "judge_time_s": round(judge_time, 1),
            "details": {
                "steps": scores.get("step_details", ""),
                "constraints": scores.get("constraint_details", ""),
                "output": scores.get("output_contract_details", ""),
            },
            "executor_response_preview": response_text[:500],
        }
        all_results.append(result)

        print(
            f"  steps={result['step_coverage']:.2f}  "
            f"constr={result['constraint_respect']:.2f}  "
            f"contract={result['output_contract_met']:.2f}  "
            f"overall={result['overall']:.2f}  "
            f"tokens={skill_tokens}  "
            f"time={exec_time:.1f}s"
        )

print(f"\nCompleted {len(all_results)} runs.")

## Format Comparison

Per-format averages sorted by overall compliance (best first, not fewest tokens).

In [ ]:
def compute_format_summary(results_list: list) -> list:
    """Compute per-format averages across all scenarios."""
    formats = {}
    for s in results_list:
        key = s["format_key"]
        if key not in formats:
            formats[key] = {"label": s["format"], "results": []}
        formats[key]["results"].append(s)

    summaries = []
    for key, data in formats.items():
        rs = data["results"]
        n = len(rs)
        summaries.append({
            "format": data["label"],
            "format_key": key,
            "count": n,
            "avg_tokens": sum(r["skill_tokens"] for r in rs) / n,
            "avg_step_coverage": sum(r["step_coverage"] for r in rs) / n,
            "avg_constraint_respect": sum(r["constraint_respect"] for r in rs) / n,
            "avg_output_contract": sum(r["output_contract_met"] for r in rs) / n,
            "avg_overall": sum(r["overall"] for r in rs) / n,
            "min_overall": min(r["overall"] for r in rs),
            "max_overall": max(r["overall"] for r in rs),
            "stddev_overall": stddev([r["overall"] for r in rs]),
        })
    summaries.sort(key=lambda x: -x["avg_overall"])
    return summaries


summaries = compute_format_summary(all_results)

print(f"{'Format':20s} {'Tokens':>7s} {'Steps':>7s} {'Constr':>8s} {'Contract':>9s} {'Overall':>8s} {'StdDev':>7s} {'Min':>5s} {'Max':>5s}")
print("-" * 90)
for s in summaries:
    print(
        f"{s['format']:20s} {s['avg_tokens']:>7.0f} {s['avg_step_coverage']:>7.2f} "
        f"{s['avg_constraint_respect']:>8.2f} {s['avg_output_contract']:>9.2f} "
        f"{s['avg_overall']:>8.2f} {s['stddev_overall']:>7.3f} {s['min_overall']:>5.2f} {s['max_overall']:>5.2f}"
    )

## Token Efficiency

Compliance-per-1K-tokens ratio: how much compliance do you get per token spent?

In [ ]:
def compute_token_efficiency(results_list: list) -> list:
    """Compute compliance-per-token ratio for each format."""
    efficiency = compute_format_summary(results_list)
    for s in efficiency:
        if s["avg_tokens"] > 0:
            s["compliance_per_1k_tokens"] = round(
                s["avg_overall"] / (s["avg_tokens"] / 1000), 4
            )
        else:
            s["compliance_per_1k_tokens"] = 0
    efficiency.sort(key=lambda x: -x["compliance_per_1k_tokens"])
    return efficiency


efficiency = compute_token_efficiency(all_results)

print(f"{'Format':20s} {'Tokens':>7s} {'Overall':>8s} {'Compliance/1K tok':>18s}")
print("-" * 58)
for e in efficiency:
    print(
        f"{e['format']:20s} {e['avg_tokens']:>7.0f} "
        f"{e['avg_overall']:>8.2f} {e['compliance_per_1k_tokens']:>18.4f}"
    )

best = efficiency[0]
print(f"\nMost efficient: {best['format']} -- {best['compliance_per_1k_tokens']:.4f} compliance per 1K tokens")

## Difficulty Breakdown

Where does the format gap show most? Group by difficulty (easy/medium/hard) and show per-format performance. The hypothesis: easy scenarios show little difference (all formats work), the gap emerges on hard scenarios that require judgment and restraint.

In [ ]:
def compute_difficulty_breakdown(results_list: list) -> dict:
    """Break down results by scenario difficulty level."""
    by_difficulty = {}
    for s in results_list:
        diff = s.get("difficulty", "unknown")
        if diff not in by_difficulty:
            by_difficulty[diff] = {}
        key = s["format_key"]
        if key not in by_difficulty[diff]:
            by_difficulty[diff][key] = {"label": s["format"], "results": []}
        by_difficulty[diff][key]["results"].append(s)

    breakdown = {}
    for diff, formats in by_difficulty.items():
        breakdown[diff] = []
        for key, data in formats.items():
            rs = data["results"]
            n = len(rs)
            breakdown[diff].append({
                "format": data["label"],
                "format_key": key,
                "avg_overall": sum(r["overall"] for r in rs) / n,
                "avg_step_coverage": sum(r["step_coverage"] for r in rs) / n,
                "avg_constraint_respect": sum(r["constraint_respect"] for r in rs) / n,
                "count": n,
            })
        breakdown[diff].sort(key=lambda x: -x["avg_overall"])
    return breakdown


by_diff = compute_difficulty_breakdown(all_results)

for diff in ["easy", "medium", "hard"]:
    if diff not in by_diff:
        continue
    print(f"\n[{diff.upper()}]")
    print(f"  {'Format':20s} {'Overall':>8s} {'Steps':>7s} {'Constr':>8s} {'n':>3s}")
    print(f"  {'-' * 50}")
    for f in by_diff[diff]:
        print(
            f"  {f['format']:20s} {f['avg_overall']:>8.2f} "
            f"{f['avg_step_coverage']:>7.2f} {f['avg_constraint_respect']:>8.2f} {f['count']:>3d}"
        )

## Pairwise Wins

For each scenario, which format scored highest? Count wins across all scenarios. Ties (within 0.01) are counted separately.

In [ ]:
def compute_pairwise_wins(results_list: list) -> dict:
    """For each scenario, which format scored highest? Count wins."""
    by_scenario = {}
    for s in results_list:
        name = s["scenario"]
        if name not in by_scenario:
            by_scenario[name] = []
        by_scenario[name].append(s)

    wins = {}
    ties = 0
    details = []
    for name, entries in by_scenario.items():
        best = max(entries, key=lambda x: x["overall"])
        best_score = best["overall"]
        winners = [e for e in entries if abs(e["overall"] - best_score) < 0.01]
        if len(winners) > 1:
            ties += 1
            details.append((name, "TIE", best_score))
        else:
            fmt = best["format"]
            wins[fmt] = wins.get(fmt, 0) + 1
            details.append((name, fmt, best_score))

    return {"wins": wins, "ties": ties, "total_scenarios": len(by_scenario), "details": details}


pw = compute_pairwise_wins(all_results)

print("Wins per format:")
print(f"  {'Format':20s} {'Wins':>5s}")
print(f"  {'-' * 28}")
for fmt, count in sorted(pw["wins"].items(), key=lambda x: -x[1]):
    bar = "#" * count
    print(f"  {fmt:20s} {count:>5d}  {bar}")
print(f"  {'(ties)':20s} {pw['ties']:>5d}")
print(f"\nTotal scenarios: {pw['total_scenarios']}")

print("\nPer-scenario winners:")
for name, winner, score in pw["details"]:
    print(f"  {name:45s}  {winner:20s}  {score:.2f}")

## Key Findings

- **LML Aggressive achieves highest overall compliance at fewer tokens** than other structured formats. Typed tags (`@step:`, `@verify:`, `@pre:`) act as explicit instruction delimiters that LLMs can latch onto.

- **The gap matters most on hard scenarios** where subtle judgment is needed. Easy scenarios (find the SQL injection) show near-perfect compliance across all formats. Hard scenarios (approve clean code without over-flagging) reveal whether the LLM internalized the skill's constraints.

- **Typed blocks help LLMs identify instruction boundaries.** `@step`, `@verify`, `@red_flag`, and `@output_contract` are unambiguous markers that prevent the LLM from conflating instructions with examples or context. In Markdown, these are just paragraphs with bold headers, easily confused with surrounding prose.

- **Instruction following is the primary value -- token savings are secondary.** A format that costs 5% more tokens but achieves 10 percentage points better compliance is the better choice for production use. AIF LML Aggressive delivers both: better compliance AND fewer tokens than the Markdown baseline.

## Save Results

In [ ]:
output = {
    "model_executor": MODEL_EXECUTOR,
    "model_judge": MODEL_JUDGE,
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "scenario_count": len(SCENARIOS),
    "format_count": len(FORMATS),
    "total_runs": len(all_results),
    "format_summary": compute_format_summary(all_results),
    "token_efficiency": compute_token_efficiency(all_results),
    "difficulty_breakdown": compute_difficulty_breakdown(all_results),
    "pairwise_wins": compute_pairwise_wins(all_results),
    "scenarios": all_results,
}

output_path = Path(os.path.abspath("")) / "results.json"
with open(output_path, "w") as f:
    json.dump(output, f, indent=2)

print(f"Results saved to {output_path}")
print(f"Total runs: {len(all_results)}")
if summaries:
    print(f"Best format: {summaries[0]['format']} (overall: {summaries[0]['avg_overall']:.2f})")